# KrishiDisha tabular models - training on Kaggle

Runs `ml.train_tabular` (crop recommendation, fertilizer on the 750k-row Playground S5E6 table, yield with the time-split protocol) on a T4 with GPU XGBoost. Outputs land in `/kaggle/working/models` (private kernel: the .pkl weights are never published; copy `metrics_tabular.json` and `reports/` back into the repo).


In [ ]:
REPO = "https://github.com/shivpratapsinghpanwar/KrishiDisha.ai.git"
BRANCH = "main"
TASKS = "fertilizer"          # e.g. "crop fertilizer yield"
import os, subprocess, glob
W = "/kaggle/working"
if not os.path.exists("/tmp/krishidisha"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/tmp/krishidisha"], check=True)
os.chdir("/tmp/krishidisha")
os.environ["KRISHIDISHA_XGB_DEVICE"] = "cuda"
FERT = (glob.glob("/kaggle/input/**/playground-series-s5e6/train.csv", recursive=True) or glob.glob("/kaggle/input/**/train.csv", recursive=True))[0]
print(FERT)


In [ ]:
# the trainer imports krishidisha.services.ml, which pulls in the Flask package: install the light web deps too
CMD = "pip install -q optuna flask flask-sqlalchemy python-dotenv 2>&1 | tail -1"
!{CMD}
CMD = f"nvidia-smi --query-gpu=name,memory.used --format=csv,noheader; free -g | head -2; python -u -m ml.train_tabular --data-dir data --output {W}/models --tasks {TASKS} --fert-data {FERT} 2>&1 | tee {W}/tabular.log | grep -v Warning | tail -40"
!{CMD}


In [ ]:
import json, shutil
m = json.load(open(f"{W}/models/metrics_tabular.json"))
print(json.dumps({k: v for k, v in m.get("fertilizer", m).items() if not isinstance(v, (list, dict))}, indent=1)[:3000])
shutil.rmtree("/tmp/krishidisha/.git", ignore_errors=True)
